# Business opportunity prioritizer

**A two-minute portfolio demo:** help an operations manager choose one automation pilot from a small backlog. Follow the sample inputs → validation → transparent ranking → recommendation → checks below.

**Status:** runnable rule-based baseline for later agent experiments; no LLM, autonomous agent, API key, network call, or third-party Python dependency. All examples are synthetic. This demonstrates Python validation, business scoping, explainable decisions, and reproducible checks.

## Run it

Read the saved outputs directly on GitHub, or clone this repository and open this notebook in a Jupyter-compatible editor with a Python 3.10+ kernel. Choose **Restart Kernel and Run All**. If you need a notebook interface, install JupyterLab in a virtual environment (`python -m pip install jupyterlab`) and launch it with `python -m jupyterlab`. Jupyter is an optional viewer; the demo code uses only the standard library.

## Decision rule

Estimated weekly hours saved = weekly task count × minutes per task × automation fraction ÷ 60. Score = estimated weekly hours saved ÷ one-time implementation hours. Higher scores win; ties use alphabetical order. This favors small pilots with a fast estimated return on effort. Inputs must include all five fields shown below; missing fields raise `KeyError`, invalid values raise `ValueError`. An empty backlog returns an empty list.


In [1]:
import math

def rank_opportunities(opportunities):
    """Rank by estimated weekly hours saved per implementation hour."""
    ranked = []
    names = set()
    for item in opportunities:
        name = item["name"]
        if not isinstance(name, str) or not name.strip() or name in names:
            raise ValueError("Use unique, non-empty names.")
        names.add(name)
        for field in ("weekly_tasks", "minutes_per_task", "automation_fraction", "build_hours"):
            value = item[field]
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"{field} must be a finite number.")
        if item["weekly_tasks"] < 0 or item["minutes_per_task"] < 0:
            raise ValueError("Task volume and duration must be non-negative.")
        if not 0 <= item["automation_fraction"] <= 1 or item["build_hours"] <= 0:
            raise ValueError("Automation fraction must be 0–1; build hours must be positive.")
        saved = item["weekly_tasks"] * item["minutes_per_task"] * item["automation_fraction"] / 60
        ranked.append(dict(item, weekly_hours_saved=saved, score=saved / item["build_hours"]))
    return sorted(ranked, key=lambda row: (-row["score"], row["name"]))


## Sample backlog and result

Edit the estimates below to compare a different backlog. The expected winner is **Meeting summaries**, with **4.0 estimated hours saved per week** and a score of **1.00**. These are calculations from assumptions, not proven business impact.


In [2]:
# Synthetic estimates, not customer data or measured savings.
opportunities = [
    dict(name="Invoice triage", weekly_tasks=100, minutes_per_task=6, automation_fraction=0.6, build_hours=12),
    dict(name="Meeting summaries", weekly_tasks=20, minutes_per_task=15, automation_fraction=0.8, build_hours=4),
    dict(name="Lead research", weekly_tasks=30, minutes_per_task=10, automation_fraction=0.5, build_hours=10),
]
ranked = rank_opportunities(opportunities)
for row in ranked:
    print(f'{row["name"]}: {row["weekly_hours_saved"]:.1f} estimated hours/week; score {row["score"]:.2f}')
print(f'Pilot recommendation: {ranked[0]["name"]}. Validate assumptions with a human owner before building.')


Meeting summaries: 4.0 estimated hours/week; score 1.00
Invoice triage: 6.0 estimated hours/week; score 0.50
Lead research: 2.5 estimated hours/week; score 0.25
Pilot recommendation: Meeting summaries. Validate assumptions with a human owner before building.


## Verification

These checks cover the expected recommendation and important boundary cases. Run them after changing the implementation.


In [3]:
assert [row["name"] for row in ranked] == ["Meeting summaries", "Invoice triage", "Lead research"]
assert ranked[0]["weekly_hours_saved"] == 4.0
assert rank_opportunities([]) == []
zero = dict(opportunities[0], weekly_tasks=0)
assert rank_opportunities([zero])[0]["score"] == 0
tie = [dict(opportunities[0], name=name) for name in ("B", "A")]
assert [row["name"] for row in rank_opportunities(tie)] == ["A", "B"]
for changes in ({"build_hours": 0}, {"automation_fraction": 1.1},
                {"weekly_tasks": -1}, {"minutes_per_task": float("nan")},
                {"weekly_tasks": True}):
    try:
        rank_opportunities([dict(opportunities[0], **changes)])
    except ValueError:
        pass
    else:
        raise AssertionError(f"Invalid input accepted: {changes}")
print("Checks passed: ranking, savings, empty input, zero volume, ties, and invalid inputs.")


Checks passed: ranking, savings, empty input, zero volume, ties, and invalid inputs.


## Limits and next experiment

The score excludes ongoing review time, maintenance, financial costs, data sensitivity, and output quality. It cannot establish demand or decide whether automation is appropriate. A human should validate the estimates and review risks before selecting a pilot.

The useful design choice here is to keep arithmetic and ranking deterministic and inspectable. A later agent could collect candidate tasks and call this function as a tool; its estimates would still need human confirmation. Once that extension is implemented and evaluated, it can become a larger project in [08-real-world-projects](../08-real-world-projects/). The eight-area learning path remains the foundation.
